<a href="https://colab.research.google.com/github/bsudharsh1979/NATcolab/blob/main/notebooks/3_ALM_Observability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NATcolab** · NVIDIA NeMo Agent Toolkit · *Asset Lifecycle Management with Agentic AI* — Google Colab edition (hosted NVIDIA NIMs)

# Asset Lifecycle Management - Observability & Monitoring

> **🟢 Colab edition.** This is the Google Colab adaptation of the NVIDIA DLI lab notebook. Instead of the lab's Docker containers it uses:
> - **Hosted NVIDIA NIMs** on build.nvidia.com — `nvidia/nemotron-3-ultra-550b-a55b` (planning, tool calling, text-to-SQL), `nvidia/nemotron-3.5-lightning-30b-a3b` (code generation), `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` (multimodal judge), `nvidia/nemotron-3.5-embed-1b` (embeddings)
> - **MOMENT-1-small**, an open-source time-series foundation model that runs locally, in place of the self-hosted NV-Tesseract NIM
> - a local Python **code sandbox** and a **Phoenix** server started from the notebook
>
> **Before you run:** add your NVIDIA API key (get one at [build.nvidia.com](https://build.nvidia.com)) as a Colab secret named `NVIDIA_API_KEY` — 🔑 icon in the left sidebar → *Add new secret* → enable *Notebook access*. A CPU runtime is enough.
> Colab gives every notebook its own VM, so run the **setup cell below at the top of every notebook** (≈5–8 min on a fresh runtime, seconds afterwards). Paths that were `/dli/task/...` in the lab are `/content/NATcolab/...` here.

> **Prerequisites:** none in Colab — the setup cell prepares everything; Phoenix and the backend are started below.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Install and start Phoenix** for observability
2. **Enable tracing** in NAT configuration
3. **Analyze traces** in the Phoenix dashboard
4. **Monitor token usage** and latency
5. **Debug agent behavior** using trace data
6. **Understand LLM calls** and tool invocations
7. **Identify performance bottlenecks** in agentic workflows

In [ ]:
# ⚙️ NATcolab setup — run first (≈5–8 min on a fresh runtime; a few seconds if already done in this session)
# Clones github.com/bsudharsh1979/NATcolab, creates the NAT virtual env (nvidia-nat 1.2.1 + ALM agent),
# builds the NASA turbofan database, trains the RUL model and checks the hosted NVIDIA NIMs.
import os, sys, subprocess
ALM_HOME = os.environ.get("ALM_HOME", "/content/NATcolab")
if not os.path.exists(f"{ALM_HOME}/colab_bootstrap.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/bsudharsh1979/NATcolab.git", ALM_HOME], check=True)
sys.path.insert(0, ALM_HOME)
os.chdir(ALM_HOME)

import colab_bootstrap
colab_bootstrap.bootstrap()

In [ ]:
from nat_client import NATClient, link
client = NATClient()

> **Reset codebase:** If you need to restore all source files and config to their original state (e.g. after testing a solution), run the cell below.

In [ ]:
client.reset()

---
# Why Observability Matters

## Challenges with Agentic AI Systems

Agentic AI systems are complex:
- Multiple LLM calls per query
- Tool invocations with external APIs
- Multi-step reasoning chains
- Iterative execution with retries

**Without observability, you can't:**
- Understand why an agent failed
- Optimize costs (token usage)
- Identify slow components
- Debug incorrect tool selections
- Monitor production performance

## What Phoenix Provides

**Phoenix** is an open-source observability platform for LLMs that tracks:

- **Traces**: Complete execution flow of each query
- **Spans**: Individual operations (LLM calls, tool invocations)
- **Token Usage**: Input/output tokens per LLM call
- **Latency**: Time spent in each operation
- **Inputs/Outputs**: Full request and response data
- **Errors**: Exceptions and failures

### Trace Example:

```
Query: "Show sensor data for unit 5 with anomalies"
├─ Reasoning Agent (2.1s, 450 tokens)
│  └─ Created plan with 3 steps
├─ Data Analysis Agent (8.7s)
│  ├─ SQL Retrieval Tool (1.2s, 150 tokens)
│  │  └─ Retrieved 192 rows
│  ├─ Anomaly Detection Tool (5.8s, 2000 tokens)
│  │  └─ Found 8 anomalies
│  └─ Plot Anomaly Tool (1.7s)
│     └─ Generated plot_12345.png
└─ Total: 10.8s, 2600 tokens
```

---
# Exercise 3.1: Phoenix Setup & Basic Monitoring

## Step 1: Verify Phoenix installation

In [ ]:
# Phoenix is installed in the NAT virtual env (arize-phoenix, pulled in by nvidia-nat-phoenix)
!{client.python_bin} -c "import phoenix; print(f'Phoenix version: {phoenix.__version__}')"

## Step 2: Start Phoenix Server

In the lab you would open a terminal and run `phoenix serve`. In Colab the client starts it in the background for you:

```bash
source /content/alm-venv/bin/activate
phoenix serve
```

In [ ]:
client.start_phoenix()

### Get Phoenix URL

In Colab the link below goes through Colab's authenticated port proxy (`google.colab.kernel.proxyPort`), so the dashboard opens in a new browser tab:

In [ ]:
link(6006, "Open Phoenix Dashboard")

## Step 3: Enable Tracing in NAT Config

Edit `configs/config-reasoning.yml` (in your file browser, this is at the top level).

Find the `general:` section near the top and **uncomment the telemetry lines** (remove the `# ` prefix):

```yaml
general:
  use_uvloop: true
  # --- EXERCISE 3.1: Uncomment the lines below to enable Phoenix tracing ---
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: asset-lifecycle-management
```

## Step 4: Restart Backend

<details>
<summary><b>📋 Solution (Colab): enable tracing programmatically</b></summary>

`exercise_solutions.enable_phoenix_tracing()` uncomments the block above (idempotent).
</details>

In [ ]:
import exercise_solutions as sol
sol.enable_phoenix_tracing()

In [ ]:
client.restart()

## Step 5: Run a Simple Query

Using the client below run:

```
What is the ground truth RUL of unit_number 20 in dataset FD001?
```

In [ ]:
response = client.query("What is the ground truth RUL of unit_number 20 in dataset FD001?")

## Step 6: View Trace in Phoenix

1. Open Phoenix dashboard (link above)
2. You should see a new trace appear
3. Click on it to explore:
   - Reasoning agent span
   - Data analysis agent span
   - SQL tool span
   - Token counts
   - Latency breakdown

**What to Look For:**
- Total execution time
- Which span took longest?
- Total tokens used
- Input/output prompts

### Check the traces from the notebook

The helper below reads the spans that NAT exported to Phoenix (via the Phoenix client inside the NAT venv) and summarizes the most recent trace:

In [ ]:
import phoenix_tools
spans = phoenix_tools.load_spans()
print(f"{len(spans)} spans in Phoenix")
phoenix_tools.summarize_latest_trace(spans)

---
# Exercise 3.2: Long Chain Query Analysis

## Objective

Use the client below to run a complex multi-step query and analyze its execution patterns.

Try the long chain query below:

```
Retrieve sensor data for unit 15 in FD001 training data, predict its remaining useful life, and plot the predicted RUL over time
```

In [ ]:
response = client.query(timeout=900, message="Retrieve sensor data for unit 15 in FD001 training data, predict its remaining useful life, and plot the predicted RUL over time")

In [ ]:
phoenix_tools.summarize_latest_trace(phoenix_tools.load_spans())

Navigate the Phoenix dashboard and answer these questions:

1. **What is the typical latency** for simple vs complex queries?
2. **Which LLM model** consumes the most tokens?
3. **What percentage of time** is spent in LLM calls vs tool execution?
4. **Are there any bottlenecks** you can identify?
5. **How does the reasoning agent's plan** correlate with actual execution?
6. **Do you see any unnecessary steps** that could be optimized?

## Phoenix Dashboard Navigation

### Main Dashboard
- **Traces Tab**: List of all queries
- **Models Tab**: Token usage by model
- **Projects**: Filter by project name

### Trace Details
- **Timeline View**: Visual span hierarchy
- **Table View**: All spans with metrics
- **Span Inspector**: Full input/output data
- **Attributes**: Metadata and tags

### Key Metrics to Monitor

1. **Latency Distribution**
   - Which operations are slowest?
   - Are there outliers?
   - Where can we optimize?

2. **Token Usage**
   - Which LLM uses most tokens?
   - Input vs output ratio
   - Cost estimation (tokens × price)

3. **Tool Invocation Patterns**
   - Are tools called in optimal order?
   - Unnecessary tool calls?
   - Failed tool calls?

4. **Error Rates**
   - How often do queries fail?
   - What causes failures?
   - Retry patterns

---
# Key Takeaways

In this notebook, you learned:

1. ✅ Why observability is critical for agentic AI systems
2. ✅ How to install and configure Phoenix
3. ✅ How to enable tracing in NAT
4. ✅ How to analyze traces in the Phoenix dashboard
5. ✅ How to identify performance bottlenecks
6. ✅ How to monitor token usage and costs
7. ✅ How to debug agent behavior using traces

## Observability Best Practices

- **Always enable tracing** in development and production
- **Monitor token costs** to optimize LLM usage
- **Track latency** to meet SLAs
- **Analyze failed queries** to improve robustness
- **Use traces for debugging** instead of print statements
- **Set up alerts** for anomalies in production

## What's Next?

In **Notebook 4**, you will:
- Understand evaluation frameworks for agentic AI
- Run NAT's built-in eval harness
- Create custom evaluation datasets
- Analyze evaluation results
- Learn to improve agent performance based on eval feedback

**[Continue to Notebook 4 - Evaluation →](4_ALM_Evaluation.ipynb)**